# Whisper-large-v3 Yorùbá — test notebook

Companion to `Whisper.ipynb` (the training notebook). This one is **inference-only**:
no Unsloth, no LoRA wiring, no training deps. It loads a merged checkpoint from HF Hub
and evaluates it. Runs comfortably on a free Colab **T4** (≈4 GB VRAM at fp16).

**What it does**
1. Load the merged 16-bit checkpoint from `MODEL_ID` (defaults to your fine-tune).
2. Single-clip sanity check on a streamed FLEURS `yo_ng` sample (with audio playback).
3. WER on N FLEURS test clips — same setup as `scripts/eval_wer.py`.
4. Optional A/B vs base `openai/whisper-large-v3` so you can quantify the gain.
5. Optional: transcribe a file you upload yourself.

**Runtime**: any GPU with ≥6 GB VRAM. Colab T4 is fine. CPU works but is slow.

### Install

In [ ]:
%%capture
!pip install -q "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.24" \
    librosa soundfile evaluate jiwer torchcodec accelerate

### HF auth (optional)

Only needed if the model repo is **private**. The fine-tune is public by default, so you
can skip this. If you do set `HF_TOKEN` in Colab Secrets (🔑 sidebar), this cell picks it up.

In [ ]:
import os

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped (public model OK)")

### Config

- `MODEL_ID` — the merged 16-bit checkpoint you pushed at the end of training.
- `BASELINE_ID` — `openai/whisper-large-v3` is the canonical baseline.
- `N_EVAL` — number of FLEURS clips. 50 for a sanity check, 200 for a real reading.

In [ ]:
MODEL_ID    = "devalade/whisper-large-v3-yoruba-colab"   # your fine-tune
BASELINE_ID = "openai/whisper-large-v3"                  # baseline for A/B
LANGUAGE    = "yoruba"
TASK        = "transcribe"
N_EVAL      = 50

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"device={DEVICE}  dtype={DTYPE}")

### Load the fine-tuned model

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE,
).to(DEVICE).eval()

# Force Yorùbá decoding — overrides whatever the checkpoint defaults to.
model.generation_config.language = "<|yo|>"
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

print(f"loaded {MODEL_ID}")

### Sanity check on one FLEURS clip

Streams a single Yorùbá clip from FLEURS, plays it back, and prints reference vs hypothesis.

In [ ]:
import soundfile as sf
from datasets import load_dataset, Audio
from IPython.display import Audio as AudioDisplay, display

fleurs = load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
sample = next(iter(fleurs.cast_column("audio", Audio(sampling_rate=16000))))
audio_path = "fleurs_yo_sample.wav"
sf.write(audio_path, sample["audio"]["array"], 16000)

ref = sample.get("transcription") or sample.get("raw_transcription") or ""

feats = processor.feature_extractor(
    sample["audio"]["array"],
    sampling_rate=16000,
    return_tensors="pt",
).input_features.to(DEVICE, dtype=DTYPE)

with torch.inference_mode():
    pred_ids = model.generate(feats, language="<|yo|>", task=TASK, max_new_tokens=256, num_beams=1)
hyp = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)[0]

print(f"Reference : {ref}")
print(f"Hypothesis: {hyp}")
display(AudioDisplay(audio_path, rate=16000))

### WER on FLEURS yo_ng (N clips)

Same setup as `scripts/eval_wer.py` — the WER number lands directly in the project's
experiments table. Run with `N_EVAL=200` for a number you can quote.

In [ ]:
import itertools
import evaluate
from tqdm.auto import tqdm
from datasets import load_dataset, Audio

wer_metric = evaluate.load("wer")
fleurs = (
    load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
    .cast_column("audio", Audio(sampling_rate=16000))
)

def transcribe(model, audio_array):
    feats = processor.feature_extractor(
        audio_array, sampling_rate=16000, return_tensors="pt",
    ).input_features.to(DEVICE, dtype=DTYPE)
    with torch.inference_mode():
        ids = model.generate(feats, language="<|yo|>", task=TASK, max_new_tokens=256, num_beams=1)
    return processor.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]

refs, hyps = [], []
for s in tqdm(itertools.islice(fleurs, N_EVAL * 2), total=N_EVAL, desc=f"WER N={N_EVAL}"):
    if len(refs) >= N_EVAL:
        break
    r = s.get("transcription") or s.get("raw_transcription") or ""
    if not r.strip():
        continue
    refs.append(r)
    hyps.append(transcribe(model, s["audio"]["array"]))

wer = 100 * wer_metric.compute(predictions=hyps, references=refs)
print(f"\n{MODEL_ID}")
print(f"FLEURS yo_ng — N={len(refs)}  WER = {wer:.2f}%")

print("\n--- samples ---")
for i in range(min(5, len(refs))):
    print(f"REF[{i}]: {refs[i]}")
    print(f"HYP[{i}]: {hyps[i]}\n")

### Normalized WER — what the model is actually doing

Raw WER on Yorùbá is brutal because of three things that have **nothing to do with model quality**:

1. **Punctuation**: Whisper inserts `, . ?` and capitalizes. FLEURS references are lowercase and unpunctuated. Every comma attached to a word is a WER error.
2. **Diacritic conventions differ** between training data and FLEURS — and even *within* FLEURS itself, references are inconsistent (some samples have full tone marks, others have none).
3. **Word boundaries**: `torípé` ↔ `to rii pe` is three "errors" for the same content.

This is exactly why the Whisper paper applies `BasicTextNormalizer` to every non-English language before computing WER. We compute three numbers:

- **Raw WER** — what you saw above. Includes all the noise.
- **Normalized WER** — lowercase, strip punctuation, collapse whitespace. **This is the comparable number** — closest to what other Yorùbá ASR papers report.
- **Permissive WER** — also strips combining marks (`ọ→o`, `ẹ→e`, `ṣ→s`, tone marks). Tells you "did the model get the *consonants and vowels* right" independent of diacritics. Upper bound on content correctness.

In [ ]:
# Normalized + permissive WER on the refs/hyps from the previous cell.
import re
import unicodedata

_PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
_WS_RE    = re.compile(r"\s+")

def normalize(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace. Keeps diacritics."""
    s = s.lower()
    s = _PUNCT_RE.sub(" ", s)
    s = _WS_RE.sub(" ", s).strip()
    return s

def strip_diacritics(s: str) -> str:
    """NFD-decompose and drop combining marks. ọ→o, ẹ→e, ṣ→s, tone marks gone."""
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)

assert refs and hyps, "Run the WER cell above first — refs/hyps must be populated."

refs_norm = [normalize(r) for r in refs]
hyps_norm = [normalize(h) for h in hyps]

refs_perm = [strip_diacritics(r) for r in refs_norm]
hyps_perm = [strip_diacritics(h) for h in hyps_norm]

wer_raw   = 100 * wer_metric.compute(predictions=hyps,      references=refs)
wer_norm  = 100 * wer_metric.compute(predictions=hyps_norm, references=refs_norm)
wer_perm  = 100 * wer_metric.compute(predictions=hyps_perm, references=refs_perm)

print(f"{MODEL_ID}  (FLEURS yo_ng, N={len(refs)})")
print(f"  raw WER         = {wer_raw:6.2f}%   ← what you saw before; noisy")
print(f"  normalized WER  = {wer_norm:6.2f}%   ← compare to published numbers")
print(f"  permissive WER  = {wer_perm:6.2f}%   ← no diacritics; content only")

print("\n--- normalized samples (what WER actually sees) ---")
for i in range(min(5, len(refs_norm))):
    print(f"REF[{i}]: {refs_norm[i]}")
    print(f"HYP[{i}]: {hyps_norm[i]}\n")

### YASR-Bench — compare all Whisper models on the same clips

Drops every model in `MODELS` through the same N FLEURS clips and reports the YASR-Bench suite. Each model is loaded, evaluated, then unloaded before the next so even a T4 can run the full comparison.

**Metrics** (all reported per-model):

- **Y-WER**: WER on normalized text (lowercase, strip punctuation, keep diacritics). The headline number.
- **Y-WER-perm**: WER on normalized + diacritic-stripped text. Content-only correctness.
- **Y-CER**: Character Error Rate on normalized text. Robust to word-boundary jitter.
- **Y-CER-perm**: CER on diacritic-stripped text.
- **diacritic gap**: Y-WER − Y-WER-perm. How much of the WER comes from diacritic mismatch alone.
- **Δ vs baseline**: improvement over `MODELS[0]` on the *same* N clips. Only metric that survives normalizer disputes.

Add a model: append its HF repo ID to `MODELS`. The first entry is the baseline that Δ-numbers are computed against.

In [ ]:
# YASR-Bench — multi-model comparison on a shared FLEURS yo_ng slice.
import gc
import itertools
import torch
import evaluate
from datasets import load_dataset, Audio
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# === The list of models to compare. First entry is the baseline for Δ. ===
MODELS = [
    "openai/whisper-large-v3",                       # baseline
    "devalade/whisper-large-v3-yoruba-colab",        # current fine-tune
    # "devalade/whisper-large-v3-yoruba-v3",         # previous fine-tune (uncomment to include)
    # "<hf-user>/<another-whisper-yoruba>",          # add more here
]

# === Cache FLEURS samples once so every model sees identical input ===
print(f"Caching {N_EVAL} FLEURS yo_ng test samples…")
fleurs_stream = (
    load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
    .cast_column("audio", Audio(sampling_rate=16000))
)
SAMPLES = []
for s in itertools.islice(fleurs_stream, N_EVAL * 2):
    ref = s.get("transcription") or s.get("raw_transcription") or ""
    if ref.strip() and len(SAMPLES) < N_EVAL:
        SAMPLES.append({"audio": s["audio"]["array"], "ref": ref})
REFS = [s["ref"] for s in SAMPLES]
print(f"  cached {len(SAMPLES)} samples")

# === Metrics ===
wer_m = evaluate.load("wer")
cer_m = evaluate.load("cer")

def yasr_bench(refs, hyps):
    refs_n = [normalize(r) for r in refs]
    hyps_n = [normalize(h) for h in hyps]
    refs_p = [strip_diacritics(r) for r in refs_n]
    hyps_p = [strip_diacritics(h) for h in hyps_n]
    return {
        "Y-WER":      100 * wer_m.compute(predictions=hyps_n, references=refs_n),
        "Y-WER-perm": 100 * wer_m.compute(predictions=hyps_p, references=refs_p),
        "Y-CER":      100 * cer_m.compute(predictions=hyps_n, references=refs_n),
        "Y-CER-perm": 100 * cer_m.compute(predictions=hyps_p, references=refs_p),
    }

# === Free the model loaded earlier in the notebook so we start clean ===
for name in ("model", "processor"):
    if name in globals():
        del globals()[name]
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# === Evaluate each model ===
results = {}
hyps_by_model = {}

for model_id in MODELS:
    print(f"\n=== {model_id} ===")
    try:
        proc = WhisperProcessor.from_pretrained(model_id)
        mdl = WhisperForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=DTYPE,
        ).to(DEVICE).eval()
        mdl.generation_config.language = "<|yo|>"
        mdl.generation_config.task = TASK
        mdl.generation_config.forced_decoder_ids = None
    except Exception as e:
        print(f"  load failed: {type(e).__name__}: {e}")
        continue

    hyps = []
    short_name = model_id.split("/")[-1]
    with torch.inference_mode():
        for s in tqdm(SAMPLES, desc=short_name):
            feats = proc.feature_extractor(
                s["audio"], sampling_rate=16000, return_tensors="pt",
            ).input_features.to(DEVICE, dtype=DTYPE)
            ids = mdl.generate(
                feats, language="<|yo|>", task=TASK,
                max_new_tokens=256, num_beams=1,
            )
            hyps.append(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

    metrics = yasr_bench(REFS, hyps)
    results[model_id] = metrics
    hyps_by_model[model_id] = hyps
    print(f"  Y-WER={metrics['Y-WER']:.2f}%  Y-WER-perm={metrics['Y-WER-perm']:.2f}%  "
          f"Y-CER={metrics['Y-CER']:.2f}%")

    del mdl, proc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# === Comparison table ===
print(f"\n\nYASR-Bench — FLEURS yo_ng, N={len(REFS)}")
print(f"baseline = {MODELS[0]}")
print("-" * 100)
header = f"{'model':<55}  {'Y-WER':>7}  {'Y-WER-perm':>10}  {'Y-CER':>7}  {'diac gap':>8}  {'Δ Y-WER':>8}"
print(header)
print("-" * 100)

baseline_wer = results.get(MODELS[0], {}).get("Y-WER")
for mid, m in results.items():
    diac_gap = m["Y-WER"] - m["Y-WER-perm"]
    if baseline_wer is None or mid == MODELS[0]:
        delta = "  —  "
    else:
        d = baseline_wer - m["Y-WER"]
        delta = f"{d:+.2f}"
    print(f"{mid:<55}  {m['Y-WER']:7.2f}  {m['Y-WER-perm']:10.2f}  "
          f"{m['Y-CER']:7.2f}  {diac_gap:8.2f}  {delta:>8}")

# === Sample side-by-side from each model ===
print("\n\n--- 3 samples per model (normalized text) ---")
for i in range(min(3, len(REFS))):
    print(f"\nREF: {normalize(REFS[i])}")
    for mid in results:
        print(f"  {mid.split('/')[-1]:<45} {normalize(hyps_by_model[mid][i])}")

### Log YASR-Bench results to Google Drive

Persist the run so we can come back later for the thesis writeup, ablation tables, or sanity checks against future fine-tunes. Writes two things to `MyDrive/yoruba-pipeline-logs/yasr-bench/`:

- `<run-id>.json` — full payload: refs, per-model hyps, metrics, environment, normalizer version. Use this when you need to re-analyse a past run.
- `runs.csv` (appended) — one row per (run, model) with the headline metrics. Drop into Sheets or pandas for trend plots.

Drive prompts you to authorize the first time; subsequent runs in the same session are silent. If Drive isn't available (running outside Colab, auth refused), the cell falls back to `./yoruba-pipeline-logs/` so the data still lands somewhere.

**Bump `NORMALIZER_VERSION` whenever you change `normalize()` or `strip_diacritics()`.** Otherwise historical CSV rows become incomparable.

In [ ]:
# Persist YASR-Bench results.
import csv
import datetime as _dt
import json
from pathlib import Path

import transformers as _tx

NORMALIZER_VERSION = "v1"          # bump if normalize() / strip_diacritics() change
LOG_DIR_REL        = "yoruba-pipeline-logs/yasr-bench"

# === Pick a log root: Drive on Colab, ./ otherwise ===
def _resolve_log_root():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        return Path("/content/drive/MyDrive") / LOG_DIR_REL, "drive"
    except Exception as e:
        print(f"  Drive unavailable ({type(e).__name__}); writing locally instead")
        return Path("./") / LOG_DIR_REL, "local"

log_root, log_kind = _resolve_log_root()
log_root.mkdir(parents=True, exist_ok=True)

run_id = _dt.datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%SZ")
json_path = log_root / f"{run_id}.json"
csv_path  = log_root / "runs.csv"

# === Gather env ===
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

# === JSON payload (full provenance for later re-analysis) ===
payload = {
    "run_id":             run_id,
    "timestamp_utc":      _dt.datetime.utcnow().isoformat() + "Z",
    "dataset":            "google/fleurs:yo_ng:test",
    "n_eval":             len(REFS),
    "language":           LANGUAGE,
    "task":               TASK,
    "normalizer_version": NORMALIZER_VERSION,
    "env": {
        "gpu":          gpu_name,
        "transformers": _tx.__version__,
        "torch":        torch.__version__,
    },
    "refs": REFS,
    "models": [
        {
            "model_id": mid,
            "metrics":  results[mid],
            "hyps":     hyps_by_model[mid],
        }
        for mid in results
    ],
}
with json_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

# === Flat CSV (one row per model per run, easy to load anywhere) ===
csv_exists = csv_path.exists()
with csv_path.open("a", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    if not csv_exists:
        w.writerow([
            "run_id", "timestamp_utc", "normalizer_version", "dataset", "n_eval",
            "model_id", "y_wer", "y_wer_perm", "y_cer", "y_cer_perm",
            "diac_gap", "gpu", "transformers_version",
        ])
    for mid, m in results.items():
        w.writerow([
            run_id, payload["timestamp_utc"], NORMALIZER_VERSION,
            payload["dataset"], len(REFS),
            mid,
            round(m["Y-WER"], 4),
            round(m["Y-WER-perm"], 4),
            round(m["Y-CER"], 4),
            round(m["Y-CER-perm"], 4),
            round(m["Y-WER"] - m["Y-WER-perm"], 4),
            gpu_name, _tx.__version__,
        ])

print(f"\nLogged to {log_kind}:")
print(f"  full run     → {json_path}")
print(f"  history CSV  → {csv_path}")
print(f"  rows appended: {len(results)}")

### Optional: transcribe your own audio

Upload a `.wav` / `.mp3` / `.flac` file (16 kHz mono works best). Useful for ad-hoc tests
of how the fine-tune handles your specific accent, recording setup, or vocabulary.

In [ ]:
RUN_UPLOAD = False  # flip True to upload

if RUN_UPLOAD:
    import librosa
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        audio, sr = librosa.load(fname, sr=16000, mono=True)
        hyp = transcribe(model, audio)
        print(f"\n{fname}")
        print(f"  → {hyp}")
else:
    print("RUN_UPLOAD=False — flip the flag and re-run to upload a file.")

## Test the v4 fine-tune from Drive\n\nLoads the merged 16-bit checkpoint that `Whisper_v4.ipynb` wrote to Drive and runs **YASR-Bench on the 50-clip holdout** — the clips the model never saw during training. This is the in-distribution overfitting check; pair it with FLEURS YASR-Bench above for the full picture.\n\nThe cell auto-discovers training runs under `MyDrive/yoruba-pipeline-logs/training/` and defaults to the most recent one. Override `RUN_ID` to pin to a specific run.\n\n**To test the v4 model on FLEURS too**: in the YASR-Bench cell above, add the same local Drive path to the `MODELS` list. `WhisperProcessor.from_pretrained` accepts local paths transparently.

In [ ]:
import gc
import json
import datetime as _dt
from pathlib import Path

import torch
from datasets import load_from_disk
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# --- Mount Drive (idempotent) and discover runs ---
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_TRAINING = Path("/content/drive/MyDrive/yoruba-pipeline-logs/training")
except Exception as e:
    DRIVE_TRAINING = Path("./yoruba-pipeline-logs/training")
    print(f"Drive unavailable ({type(e).__name__}); using local fallback")

# --- Pick the run to test ---
RUN_ID = None     # e.g. "2026-06-14T16-32-09Z" — None auto-picks latest with merged_16bit
if not DRIVE_TRAINING.exists():
    raise FileNotFoundError(f"No training runs at {DRIVE_TRAINING}. Run Whisper_v4.ipynb first.")

if RUN_ID is None:
    candidates = sorted(
        [p for p in DRIVE_TRAINING.iterdir() if (p / "merged_16bit").exists()],
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(
            f"No run with a merged_16bit/ subdir found in {DRIVE_TRAINING}. "
            "Re-run Whisper_v4.ipynb with SAVE_MERGED_DRIVE=True."
        )
    RUN_DIR = candidates[0]
    RUN_ID  = RUN_DIR.name
else:
    RUN_DIR = DRIVE_TRAINING / RUN_ID

print(f"Run        : {RUN_ID}")
print(f"Run dir    : {RUN_DIR}")

# Show training config so we know what we're testing
config_path = RUN_DIR / "config.json"
if config_path.exists():
    with config_path.open() as f:
        run_cfg = json.load(f)
    print(f"Trained on : {run_cfg['dataset']['repo']}  "
          f"(holdout={run_cfg['splits']['holdout']}, "
          f"train={run_cfg['splits']['train']}, eval={run_cfg['splits']['eval']})")
    print(f"Epochs     : up to {run_cfg['trainer']['num_epochs']}")
    print(f"Base model : {run_cfg['base_model']}")

# --- Load holdout dataset (50 clips never seen during training) ---
holdout_path = RUN_DIR / "holdout"
holdout = load_from_disk(str(holdout_path))
HOLDOUT_REFS = [r["text"] for r in holdout]
print(f"\nHoldout    : {len(holdout)} clips (these were NEVER seen during training)")

# --- Load the v4 fine-tune from Drive ---
MODEL_PATH = str(RUN_DIR / "merged_16bit")
print(f"\nLoading model from {MODEL_PATH}…")
proc_v4 = WhisperProcessor.from_pretrained(MODEL_PATH)
mdl_v4  = WhisperForConditionalGeneration.from_pretrained(
    MODEL_PATH, torch_dtype=DTYPE,
).to(DEVICE).eval()
mdl_v4.generation_config.language = "<|yo|>"
mdl_v4.generation_config.task = TASK
mdl_v4.generation_config.forced_decoder_ids = None

# --- Transcribe the holdout ---
hyps_v4 = []
with torch.inference_mode():
    for row in tqdm(holdout, desc=f"holdout v4"):
        feats = proc_v4.feature_extractor(
            row["audio"]["array"],
            sampling_rate=row["audio"]["sampling_rate"],
            return_tensors="pt",
        ).input_features.to(DEVICE, dtype=DTYPE)
        ids = mdl_v4.generate(
            feats, language="<|yo|>", task=TASK,
            max_new_tokens=256, num_beams=1,
        )
        hyps_v4.append(proc_v4.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

# --- YASR-Bench (reuses the metrics + normalize/strip_diacritics from above) ---
holdout_metrics = yasr_bench(HOLDOUT_REFS, hyps_v4)

print(f"\n=== {RUN_ID} on holdout (N={len(HOLDOUT_REFS)}) ===")
print(f"  Y-WER       = {holdout_metrics['Y-WER']:.2f}%")
print(f"  Y-WER-perm  = {holdout_metrics['Y-WER-perm']:.2f}%")
print(f"  Y-CER       = {holdout_metrics['Y-CER']:.2f}%")
print(f"  Y-CER-perm  = {holdout_metrics['Y-CER-perm']:.2f}%")
print(f"  diac gap    = {holdout_metrics['Y-WER'] - holdout_metrics['Y-WER-perm']:.2f}")

# --- Persist this eval next to the training artifacts ---
test_payload = {
    "timestamp_utc":      _dt.datetime.utcnow().isoformat() + "Z",
    "run_id":             RUN_ID,
    "dataset":            "holdout (in-distribution)",
    "n_eval":             len(HOLDOUT_REFS),
    "normalizer_version": NORMALIZER_VERSION,
    "metrics":            holdout_metrics,
    "refs":               HOLDOUT_REFS,
    "hyps":               hyps_v4,
}
with (RUN_DIR / "holdout_yasr_bench.json").open("w", encoding="utf-8") as f:
    json.dump(test_payload, f, ensure_ascii=False, indent=2)
print(f"\nwrote {RUN_DIR/'holdout_yasr_bench.json'}")

# Free VRAM
del mdl_v4, proc_v4
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# --- Show 5 samples side by side ---
print("\n--- 5 holdout samples ---")
for i in range(min(5, len(HOLDOUT_REFS))):
    print(f"REF[{i}]: {normalize(HOLDOUT_REFS[i])}")
    print(f"HYP[{i}]: {normalize(hyps_v4[i])}\n")

### Baseline `whisper-large-v3` on the same holdout — paired Δ\n\nRuns vanilla `openai/whisper-large-v3` (no fine-tuning) on the exact same 50 clips, then computes ΔY-WER, ΔY-CER, etc. against the v4 fine-tune.\n\n**Why this matters for the thesis**: the holdout's references are inconsistently diacritized, which inflates raw Y-WER for *any* model that produces diacritics. But Δ-numbers cancel that out — both models are scored against the same imperfect references, so the comparison is fair. The paired Δ is the most defensible "fine-tuning helped" number you can quote.\n\nRequires: the previous cell ran (so `holdout`, `HOLDOUT_REFS`, `hyps_v4`, `holdout_metrics`, `RUN_DIR` are in scope).

In [ ]:
import gc
import json
import datetime as _dt

import torch
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASELINE_ID = "openai/whisper-large-v3"

# Defensive — fail loudly if the previous cell didn't run
assert "holdout" in dir() and "HOLDOUT_REFS" in dir() and "hyps_v4" in dir(), \
    "Run the v4-holdout cell above first; this one reuses its state."
assert "holdout_metrics" in dir() and "RUN_DIR" in dir(), \
    "Run the v4-holdout cell above first; this one reuses its state."

print(f"Loading baseline {BASELINE_ID}…")
proc_b = WhisperProcessor.from_pretrained(BASELINE_ID)
mdl_b  = WhisperForConditionalGeneration.from_pretrained(
    BASELINE_ID, torch_dtype=DTYPE,
).to(DEVICE).eval()
mdl_b.generation_config.language = "<|yo|>"
mdl_b.generation_config.task = TASK
mdl_b.generation_config.forced_decoder_ids = None

# Same 50 clips, same decoding settings as the v4 pass
hyps_baseline = []
with torch.inference_mode():
    for row in tqdm(holdout, desc="holdout baseline"):
        feats = proc_b.feature_extractor(
            row["audio"]["array"],
            sampling_rate=row["audio"]["sampling_rate"],
            return_tensors="pt",
        ).input_features.to(DEVICE, dtype=DTYPE)
        ids = mdl_b.generate(
            feats, language="<|yo|>", task=TASK,
            max_new_tokens=256, num_beams=1,
        )
        hyps_baseline.append(proc_b.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

# --- Score baseline + compute paired Δ vs v4 ---
baseline_metrics = yasr_bench(HOLDOUT_REFS, hyps_baseline)

deltas = {k: baseline_metrics[k] - holdout_metrics[k] for k in baseline_metrics}
# Δ > 0 means baseline is worse (higher error), i.e. fine-tune wins by Δ pp

print(f"\n=== holdout (N={len(HOLDOUT_REFS)}) ===\n")
print(f"{'metric':<14}  {'baseline':>10}  {'v4 (' + RUN_ID[:10] + ')':>20}  {'Δ (baseline−v4)':>17}")
print("-" * 70)
for k in ("Y-WER", "Y-WER-perm", "Y-CER", "Y-CER-perm"):
    print(f"{k:<14}  {baseline_metrics[k]:>9.2f}%  {holdout_metrics[k]:>19.2f}%  {deltas[k]:>+16.2f}")
print(f"{'diac gap':<14}  "
      f"{baseline_metrics['Y-WER']-baseline_metrics['Y-WER-perm']:>9.2f}   "
      f"{holdout_metrics['Y-WER']-holdout_metrics['Y-WER-perm']:>19.2f}   "
      f"{(baseline_metrics['Y-WER']-baseline_metrics['Y-WER-perm']) - (holdout_metrics['Y-WER']-holdout_metrics['Y-WER-perm']):>+16.2f}")

# --- Persist the paired comparison ---
paired_payload = {
    "timestamp_utc":      _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "run_id":             RUN_ID,
    "dataset":            "holdout (in-distribution)",
    "n_eval":             len(HOLDOUT_REFS),
    "normalizer_version": NORMALIZER_VERSION,
    "baseline_model":     BASELINE_ID,
    "fine_tune_path":     str(RUN_DIR / "merged_16bit"),
    "metrics": {
        BASELINE_ID:   baseline_metrics,
        RUN_ID:        holdout_metrics,
    },
    "deltas_baseline_minus_v4": deltas,
    "refs":               HOLDOUT_REFS,
    "hyps_baseline":      hyps_baseline,
    "hyps_v4":            hyps_v4,
}
out_path = RUN_DIR / "holdout_baseline_paired.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(paired_payload, f, ensure_ascii=False, indent=2)
print(f"\nwrote {out_path}")

del mdl_b, proc_b
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# --- Three-way side-by-side: REF / baseline / v4 ---
print("\n--- 5 holdout samples (normalized) ---")
for i in range(min(5, len(HOLDOUT_REFS))):
    print(f"REF      : {normalize(HOLDOUT_REFS[i])}")
    print(f"baseline : {normalize(hyps_baseline[i])}")
    print(f"v4       : {normalize(hyps_v4[i])}\n")

### OOD benchmark — public Yorùbá corpus not in training (paired baseline vs v4)\n\nThe holdout proved the model generalizes within Hidi-agili. This cell tests **out-of-distribution audio from a different corpus** — the actual claim the thesis needs to make ("our fine-tune transfers to speech we never saw during training, in any form").\n\nDefaults to **`mozilla-foundation/common_voice_17_0` yo** — multi-speaker, fully open, not in your training mix (you have `scripts/cv_probe.py` but the v3 training config never included it). Common Voice may be **gated**: accept the dataset terms at https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0 first (one-time, free).\n\nAlternative: OpenSLR86 (Lagos University Yorùbá corpus). Commented block below — just uncomment.\n\nProduces the same paired baseline-vs-v4 table as the holdout cell, and writes `RUN_DIR/ood_<dataset>_paired.json`.

In [ ]:
import gc
import itertools
import json
import datetime as _dt

import torch
import evaluate
from datasets import load_dataset, Audio
from tqdm.auto import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# ----- Pick the OOD corpus (uncomment exactly one) ---------------------------
# A) Mozilla Common Voice yo  — recommended; multi-speaker, free, gated
OOD_DATASET   = "mozilla-foundation/common_voice_17_0"
OOD_CONFIG    = "yo"
OOD_SPLIT     = "test"
OOD_TEXT_COL  = "sentence"

# B) OpenSLR86  — Lagos University Yorùbá corpus
# OOD_DATASET   = "openslr/openslr"
# OOD_CONFIG    = "SLR86"
# OOD_SPLIT     = "train"
# OOD_TEXT_COL  = "transcription"

N_OOD            = 100        # bump to 200+ for the final thesis number
BASELINE_OOD_ID  = "openai/whisper-large-v3"

# ----- Cache OOD samples once so every model sees identical input ------------
print(f"Streaming {OOD_DATASET} ({OOD_CONFIG}) split={OOD_SPLIT}…")
try:
    ood_stream = (
        load_dataset(OOD_DATASET, OOD_CONFIG, split=OOD_SPLIT, streaming=True)
        .cast_column("audio", Audio(sampling_rate=16000))
    )
except Exception as e:
    raise RuntimeError(
        f"Couldn't load {OOD_DATASET}: {type(e).__name__}: {e}\n"
        f"If this is a gated dataset, visit "
        f"https://huggingface.co/datasets/{OOD_DATASET} and accept the terms first."
    ) from e

OOD_SAMPLES = []
for s in itertools.islice(ood_stream, N_OOD * 2):
    ref = s.get(OOD_TEXT_COL) or s.get("text") or s.get("transcription") or ""
    if ref.strip() and len(OOD_SAMPLES) < N_OOD:
        OOD_SAMPLES.append({"audio": s["audio"]["array"], "ref": ref})
OOD_REFS = [s["ref"] for s in OOD_SAMPLES]
print(f"  cached {len(OOD_SAMPLES)} samples")

# ----- Sanity: the OOD corpus must not be in training ------------------------
training_repo = run_cfg.get("dataset", {}).get("repo", "") if "run_cfg" in dir() else ""
if training_repo and OOD_DATASET.lower() in training_repo.lower():
    raise RuntimeError(
        f"OOD_DATASET={OOD_DATASET} overlaps with training repo {training_repo}. "
        "Pick a different OOD corpus."
    )

# ----- Run a model over the cached samples -----------------------------------
def _eval_model(model_id_or_path: str, short_name: str):
    print(f"\n  loading {short_name} from {model_id_or_path}")
    proc = WhisperProcessor.from_pretrained(model_id_or_path)
    mdl  = WhisperForConditionalGeneration.from_pretrained(
        model_id_or_path, torch_dtype=DTYPE,
    ).to(DEVICE).eval()
    mdl.generation_config.language = "<|yo|>"
    mdl.generation_config.task = TASK
    mdl.generation_config.forced_decoder_ids = None

    hyps = []
    with torch.inference_mode():
        for s in tqdm(OOD_SAMPLES, desc=short_name):
            feats = proc.feature_extractor(
                s["audio"], sampling_rate=16000, return_tensors="pt",
            ).input_features.to(DEVICE, dtype=DTYPE)
            ids = mdl.generate(feats, language="<|yo|>", task=TASK,
                               max_new_tokens=256, num_beams=1)
            hyps.append(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])
    del mdl, proc
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return hyps

V4_PATH = str(RUN_DIR / "merged_16bit")

hyps_b_ood  = _eval_model(BASELINE_OOD_ID, "baseline")
hyps_v4_ood = _eval_model(V4_PATH,         "v4")

# ----- Score + paired Δ ------------------------------------------------------
m_b  = yasr_bench(OOD_REFS, hyps_b_ood)
m_v4 = yasr_bench(OOD_REFS, hyps_v4_ood)
deltas_ood = {k: m_b[k] - m_v4[k] for k in m_b}

short_dataset = OOD_DATASET.split("/")[-1]
print(f"\n=== OOD: {OOD_DATASET} ({OOD_CONFIG}, {OOD_SPLIT}, N={len(OOD_REFS)}) ===\n")
print(f"{'metric':<14}  {'baseline':>10}  {'v4 (' + RUN_ID[:10] + ')':>20}  {'Δ (baseline−v4)':>17}")
print("-" * 70)
for k in ("Y-WER", "Y-WER-perm", "Y-CER", "Y-CER-perm"):
    print(f"{k:<14}  {m_b[k]:>9.2f}%  {m_v4[k]:>19.2f}%  {deltas_ood[k]:>+16.2f}")
print(f"{'diac gap':<14}  "
      f"{m_b['Y-WER']-m_b['Y-WER-perm']:>9.2f}   "
      f"{m_v4['Y-WER']-m_v4['Y-WER-perm']:>19.2f}   "
      f"{(m_b['Y-WER']-m_b['Y-WER-perm']) - (m_v4['Y-WER']-m_v4['Y-WER-perm']):>+16.2f}")

# ----- Persist ---------------------------------------------------------------
payload = {
    "timestamp_utc":      _dt.datetime.now(_dt.timezone.utc).isoformat(),
    "run_id":             RUN_ID,
    "dataset":            f"{OOD_DATASET}:{OOD_CONFIG}:{OOD_SPLIT}",
    "n_eval":             len(OOD_REFS),
    "normalizer_version": NORMALIZER_VERSION,
    "baseline_model":     BASELINE_OOD_ID,
    "fine_tune_path":     V4_PATH,
    "metrics": {
        BASELINE_OOD_ID: m_b,
        RUN_ID:          m_v4,
    },
    "deltas_baseline_minus_v4": deltas_ood,
    "refs":          OOD_REFS,
    "hyps_baseline": hyps_b_ood,
    "hyps_v4":       hyps_v4_ood,
}
slug = short_dataset.replace("_", "-")
out_path = RUN_DIR / f"ood_{slug}_paired.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print(f"\nwrote {out_path}")

# ----- 5 three-way samples ---------------------------------------------------
print("\n--- 5 OOD samples (normalized) ---")
for i in range(min(5, len(OOD_REFS))):
    print(f"REF      : {normalize(OOD_REFS[i])}")
    print(f"baseline : {normalize(hyps_b_ood[i])}")
    print(f"v4       : {normalize(hyps_v4_ood[i])}\n")